**Load Bronze Incremental Table**

In [0]:
bronze_df = spark.table("ecommerce.bronze.incremental_orders")
display(bronze_df)

order_id,order_date,customer_name,product,category,quantity,price,_rescued_data,source_file,bronze_ingested_at,discount_percentage
2021,2026-09-07,Ravi,Phone,Electronics,1,650,null,orders_day3.csv,2026-10-09T16:29:03.506Z,10
2022,2026-09-07,John,Shoes,Fashion,2,60,null,orders_day3.csv,2026-10-09T16:29:03.506Z,5
2023,2026-09-07,Priya,Laptop,Electronics,1,800,null,orders_day3.csv,2026-10-09T16:29:03.506Z,15
2024,2026-09-07,David,T-Shirt,Fashion,4,20,null,orders_day3.csv,2026-10-09T16:29:03.506Z,0
2025,2026-09-07,Anitha,Keyboard,Electronics,1,45,null,orders_day3.csv,2026-10-09T16:29:03.506Z,8
2026,2026-09-07,Robert,Backpack,Fashion,2,35,null,orders_day3.csv,2026-10-09T16:29:03.506Z,12
2027,2026-09-07,Suresh,Blender,Home,1,70,null,orders_day3.csv,2026-10-09T16:29:03.506Z,10
2028,2026-09-07,Emily,Mouse,Electronics,2,25,null,orders_day3.csv,2026-10-09T16:29:03.506Z,5
2029,2026-09-07,Arun,Headphones,Electronics,1,75,null,orders_day3.csv,2026-10-09T16:29:03.506Z,7
2030,2026-09-07,Sarah,Coffee Maker,Home,2,90,null,orders_day3.csv,2026-10-09T16:29:03.506Z,10


**Create Silver Table**

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS ecommerce.silver.incremental_orders_silver (
    order_id INT,
    order_date DATE,
    customer_name STRING,
    product STRING,
    category STRING,
    quantity INT,
    price DOUBLE,
    ingested_at TIMESTAMP
)
USING DELTA
""")


DataFrame[]

**Prepare Bronze Data**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

window_spec = (
    Window
    .partitionBy("order_id")
    .orderBy(
        F.col("bronze_ingested_at").desc(),
        F.col("source_file").desc()
    )
)

bronze_clean_df = (
    bronze_df
    .withColumn(
        "row_num",
        F.row_number().over(window_spec)
    )
    .filter(F.col("row_num") == 1)
    .drop("row_num")
    .withColumn("ingested_at", F.current_timestamp())
    .select(
        "order_id",
        "order_date",
        "customer_name",
        "product",
        "category",
        "quantity",
        "price",
        "discount_percentage",
        "ingested_at"
    )
)

display(bronze_clean_df)

order_id,order_date,customer_name,product,category,quantity,price,discount_percentage,ingested_at
2001,2026-09-05,Ravi,Laptop,Electronics,1,800,null,2026-10-09T16:50:08.089Z
2002,2026-09-06,John,Mouse,Electronics,2,25,null,2026-10-09T16:50:08.089Z
2003,2026-09-06,Priya,Shoes,Fashion,1,60,null,2026-10-09T16:50:08.089Z
2004,2026-09-05,David,Keyboard,Electronics,1,45,null,2026-10-09T16:50:08.089Z
2005,2026-09-05,Anitha,T-Shirt,Fashion,2,20,null,2026-10-09T16:50:08.089Z
2006,2026-09-05,Robert,Headphones,Electronics,1,75,null,2026-10-09T16:50:08.089Z
2007,2026-09-05,Suresh,Jeans,Fashion,1,40,null,2026-10-09T16:50:08.089Z
2008,2026-09-05,Emily,Coffee Maker,Home,1,90,null,2026-10-09T16:50:08.089Z
2009,2026-09-05,Arun,Monitor,Electronics,1,250,null,2026-10-09T16:50:08.089Z
2010,2026-09-05,Sarah,Backpack,Fashion,1,35,null,2026-10-09T16:50:08.089Z


**MERGE Bronze → Silver (UPSERT)**

In [0]:
from delta.tables import DeltaTable

silver_table = DeltaTable.forName(spark, "ecommerce.silver.incremental_orders_silver")

silver_table.alias("s").merge(
    bronze_clean_df.alias("b"),
    "s.order_id = b.order_id"
).withSchemaEvolution() \
 .whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

**Load Silver Table**

In [0]:
silver_df = spark.table("ecommerce.silver.incremental_orders_silver")
display(silver_df)

order_id,order_date,customer_name,product,category,quantity,price,ingested_at,discount_percentage
2001,2026-09-05,Ravi,Laptop,Electronics,1,800.0,2026-10-09T16:50:19.501Z,null
2002,2026-09-06,John,Mouse,Electronics,2,25.0,2026-10-09T16:50:19.501Z,null
2003,2026-09-06,Priya,Shoes,Fashion,1,60.0,2026-10-09T16:50:19.501Z,null
2004,2026-09-05,David,Keyboard,Electronics,1,45.0,2026-10-09T16:50:19.501Z,null
2005,2026-09-05,Anitha,T-Shirt,Fashion,2,20.0,2026-10-09T16:50:19.501Z,null
2006,2026-09-05,Robert,Headphones,Electronics,1,75.0,2026-10-09T16:50:19.501Z,null
2007,2026-09-05,Suresh,Jeans,Fashion,1,40.0,2026-10-09T16:50:19.501Z,null
2008,2026-09-05,Emily,Coffee Maker,Home,1,90.0,2026-10-09T16:50:19.501Z,null
2009,2026-09-05,Arun,Monitor,Electronics,1,250.0,2026-10-09T16:50:19.501Z,null
2010,2026-09-05,Sarah,Backpack,Fashion,1,35.0,2026-10-09T16:50:19.501Z,null


**Record Count Check**

In [0]:
print("Total Records in Silver Table:", silver_df.count())

Total Records in Silver Table: 30


**Check Updated Records (Example: order_id = 2002)**

In [0]:
display(silver_df.filter("order_id = 2002"))

order_id,order_date,customer_name,product,category,quantity,price,ingested_at
2002,2026-09-06,John,Mouse,Electronics,2,25.0,2026-10-09T15:59:18.258Z


**Check Schema Evolution (Day3)**

In [0]:
silver_df.printSchema()

In [0]:
display(silver_df.filter("discount_percentage IS NOT NULL"))

**Delta History (Silver Table)**

In [0]:
spark.sql("""
DESCRIBE HISTORY ecommerce.silver.incremental_orders_silver
""").show(truncate=False)


+-------+-------------------+--------------+-----------------------------+------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+----------------+------------------------------------+------------------------+-----------+-----------------+-------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------